In [1]:
#cell1
# Install runtime dependencies for GPT-OSS + vLLM on Colab / Blackwell.

!pip -q install -U uv

!uv pip install --system -U openai requests tqdm psutil numpy accelerate safetensors huggingface_hub gpt-oss openai-harmony

# Remove optional packages that may break imports or pull mismatched CUDA wheels.
!uv pip uninstall --system -y torchcodec torchvision torchaudio sentence-transformers || true

# Transformers is useful for tokenizer checks and compatibility.
!uv pip install --system -U "transformers>=4.56.0"

# GPT-OSS support is available in recent vLLM releases.
# For CUDA 13 / Blackwell Colab, try the cu130 path first.
!uv pip install --system -U vllm --torch-backend=cu130 --extra-index-url https://wheels.vllm.ai/nightly/cu130 || \
 uv pip install --system -U vllm --torch-backend=auto

# Optional fallback only if the lines above fail:
# !uv pip install --system --pre -U "vllm==0.10.1+gptoss" \
#     --extra-index-url https://wheels.vllm.ai/gpt-oss/ \
#     --extra-index-url https://download.pytorch.org/whl/nightly/cu128 \
#     --index-strategy unsafe-best-match

import sys
import importlib.metadata as md

import torch
import vllm
import transformers

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)
print("transformers:", transformers.__version__)

for pkg in ["gpt-oss", "openai-harmony", "torchcodec", "torchvision", "torchaudio", "sentence-transformers"]:
    try:
        print(pkg + ":", md.version(pkg))
    except Exception:
        print(pkg + ": not installed")

!nvidia-smi

Using Python 3.12.13 environment at: /usr
Resolved 93 packages in 154ms
Prepared 8 packages in 0.34ms
Uninstalled 8 packages in 124ms
Installed 8 packages in 115ms
 - numpy==2.3.5
 + numpy==2.4.6
 - nvidia-cublas==13.1.0.3
 + nvidia-cublas==13.1.1.3
 - nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cudnn-cu13==9.20.0.48
 - nvidia-cusparselt-cu13==0.8.0
 + nvidia-cusparselt-cu13==0.8.1
 - nvidia-nccl-cu13==2.28.9
 + nvidia-nccl-cu13==2.29.7
 - setuptools==80.10.2
 + setuptools==81.0.0
 - torch==2.11.0+cu130
 + torch==2.12.0
 - triton==3.6.0
 + triton==3.7.0
Using Python 3.12.13 environment at: /usr
Uninstalled 2 packages in 47ms
 - torchaudio==2.11.0+cu130
 - torchvision==0.26.0+cu130
Using Python 3.12.13 environment at: /usr
Resolved 27 packages in 73ms
Checked 27 packages in 0.24ms
Using Python 3.12.13 environment at: /usr
Resolved 189 packages in 6.76s
Prepared 10 packages in 13ms
Uninstalled 8 packages in 108ms
Installed 10 packages in 109ms
 - numpy==2.4.6
 + numpy==2.3.5
 - nvidia-cublas=

In [2]:
#cell2
# Imports and global config.

import os
import re
import gc
import json
import time
import shlex
import shutil
import psutil
import subprocess
import traceback
import site
import glob

from pathlib import Path
from tqdm.auto import tqdm
from openai import OpenAI

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# GPT-OSS model.
LLM_MODEL_NAME = "openai/gpt-oss-120b"

# GPT-OSS supports reasoning effort levels: low, medium, high.
# Keep high to match your previous GPT-OSS notebook.
REASONING_EFFORT = "high"

PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# No context is provided, but GPT-OSS may spend tokens on reasoning.
# 4096 keeps the server lighter than full 131072 context.
MAX_MODEL_LEN = 8192

GPU_MEMORY_UTILIZATION = 0.95

MAX_NUM_SEQS = 1

# Recommended GPT-OSS value in vLLM recipes.
# If startup OOM happens, try 4096, then 2048, then 1024.
MAX_NUM_BATCHED_TOKENS = 16384

# Useful on Blackwell. If startup fails on your GPU, set this to None.
KV_CACHE_DTYPE = "fp8"

ENABLE_PREFIX_CACHING = False
MAX_CUDAGRAPH_CAPTURE_SIZE = 2048

# This avoids FlashInfer sampler issues that can happen in some Colab/CUDA setups.
USE_FLASHINFER_SAMPLER = False

# Use this only if the server still crashes inside FlashInfer.
DISABLE_FLASHINFER_COMPLETELY = False

# Optional fallback. Keep None first.
# If attention backend errors happen, you can try: "TRITON_ATTN"
ATTENTION_BACKEND = None

ADD_NVIDIA_PIP_LIBS_TO_LD_LIBRARY_PATH = True

SERVER_LOG_PATH = Path("/content/vllm_gpt_oss_120b_no_context_server.log")
SERVER_PID_PATH = Path("/content/vllm_gpt_oss_120b_no_context_server.pid")

LOCAL_RUNTIME_DIR = Path("/content/final_project_copy")
LOCAL_EVIDENCE_DIR = LOCAL_RUNTIME_DIR / "RAG" / "evidence"
LOCAL_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/final_project")
DRIVE_EVIDENCE_DIR = DRIVE_PROJECT_DIR / "RAG" / "evidence"
DRIVE_LLM_OUTPUT_DIR = DRIVE_PROJECT_DIR / "LLM"

DATASETS = [
    {
        "name": "2wikimultihopqa",
        "evidence_filename": "2wikimultihopqa_evidence.json",
        "answer_filename": "2wikimultihopqa_gpt_oss_120b_answers.json",
    },
    {
        "name": "hotpotqa",
        "evidence_filename": "hotpotqa_evidence.json",
        "answer_filename": "hotpotqa_gpt_oss_120b_answers.json",
    },
]

ANSWER_START_INDEX = 0
ANSWER_END_INDEX = None  # None means all records.

SAVE_EVERY_N = 1
CLEAR_CACHE_EVERY_N = 25

# Final answer is short, but GPT-OSS may use reasoning tokens before final text.
ANSWER_MAX_TOKENS = 4096
ANSWER_RETRY_MAX_TOKENS = 4096

# User requested less than 6 words, so this means at most 5 words.
MAX_ANSWER_WORDS = 5

UNKNOWN_ANSWER = "I don't know"

print("Model:", LLM_MODEL_NAME)
print("Reasoning effort:", REASONING_EFFORT)
print("Max model len:", MAX_MODEL_LEN)
print("GPU memory utilization:", GPU_MEMORY_UTILIZATION)
print("Max num batched tokens:", MAX_NUM_BATCHED_TOKENS)
print("KV cache dtype:", KV_CACHE_DTYPE)
print("Prefix caching enabled:", ENABLE_PREFIX_CACHING)
print("Use FlashInfer sampler:", USE_FLASHINFER_SAMPLER)
print("Disable FlashInfer completely:", DISABLE_FLASHINFER_COMPLETELY)
print("Attention backend override:", ATTENTION_BACKEND)
print("Drive evidence directory:", DRIVE_EVIDENCE_DIR)
print("Drive LLM output directory:", DRIVE_LLM_OUTPUT_DIR)

for ds in DATASETS:
    print("-" * 80)
    print("Dataset:", ds["name"])
    print("Evidence:", DRIVE_EVIDENCE_DIR / ds["evidence_filename"])
    print("Output:", DRIVE_LLM_OUTPUT_DIR / ds["answer_filename"])

Model: openai/gpt-oss-120b
Reasoning effort: high
Max model len: 8192
GPU memory utilization: 0.95
Max num batched tokens: 16384
KV cache dtype: fp8
Prefix caching enabled: False
Use FlashInfer sampler: False
Disable FlashInfer completely: False
Attention backend override: None
Drive evidence directory: /content/drive/MyDrive/final_project/RAG/evidence
Drive LLM output directory: /content/drive/MyDrive/final_project/LLM
--------------------------------------------------------------------------------
Dataset: 2wikimultihopqa
Evidence: /content/drive/MyDrive/final_project/RAG/evidence/2wikimultihopqa_evidence.json
Output: /content/drive/MyDrive/final_project/LLM/2wikimultihopqa_gpt_oss_120b_answers.json
--------------------------------------------------------------------------------
Dataset: hotpotqa
Evidence: /content/drive/MyDrive/final_project/RAG/evidence/hotpotqa_evidence.json
Output: /content/drive/MyDrive/final_project/LLM/hotpotqa_gpt_oss_120b_answers.json


In [3]:
#cell3
# Mount Google Drive and copy both evidence files to local Colab disk.

from google.colab import drive

MOUNTPOINT = Path("/content/drive")
drive.mount(str(MOUNTPOINT), force_remount=True)

DRIVE_LLM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

def file_is_same_size(src: Path, dst: Path) -> bool:
    # Check whether local copy is complete.
    return dst.exists() and dst.stat().st_size == src.stat().st_size

def copy_file_to_local(src: Path, dst: Path) -> None:
    # Copy with temp file to avoid partial local copies.
    dst.parent.mkdir(parents=True, exist_ok=True)

    if file_is_same_size(src, dst):
        print("Local copy already exists:", dst)
        return

    tmp = dst.with_name(dst.name + ".tmp")

    if tmp.exists():
        tmp.unlink()

    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

for ds in DATASETS:
    drive_evidence_path = DRIVE_EVIDENCE_DIR / ds["evidence_filename"]
    local_evidence_path = LOCAL_EVIDENCE_DIR / ds["evidence_filename"]

    if not drive_evidence_path.exists():
        raise FileNotFoundError(f"Evidence file not found: {drive_evidence_path}")

    copy_file_to_local(drive_evidence_path, local_evidence_path)

    ds["drive_evidence_path"] = drive_evidence_path
    ds["local_evidence_path"] = local_evidence_path
    ds["drive_answer_path"] = DRIVE_LLM_OUTPUT_DIR / ds["answer_filename"]

    print("=" * 100)
    print("Dataset:", ds["name"])
    print("Drive evidence:", drive_evidence_path)
    print("Local evidence:", local_evidence_path)
    print("Local evidence size MB:", local_evidence_path.stat().st_size / (1024 ** 2))
    print("Output path:", ds["drive_answer_path"])

Mounted at /content/drive
Dataset: 2wikimultihopqa
Drive evidence: /content/drive/MyDrive/final_project/RAG/evidence/2wikimultihopqa_evidence.json
Local evidence: /content/final_project_copy/RAG/evidence/2wikimultihopqa_evidence.json
Local evidence size MB: 5.524371147155762
Output path: /content/drive/MyDrive/final_project/LLM/2wikimultihopqa_gpt_oss_120b_answers.json
Dataset: hotpotqa
Drive evidence: /content/drive/MyDrive/final_project/RAG/evidence/hotpotqa_evidence.json
Local evidence: /content/final_project_copy/RAG/evidence/hotpotqa_evidence.json
Local evidence size MB: 6.91798210144043
Output path: /content/drive/MyDrive/final_project/LLM/hotpotqa_gpt_oss_120b_answers.json


In [4]:
#cell4
# Load both evidence datasets.

def load_evidence_records(path: Path):
    # Load a JSON list of evidence records.
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    if not isinstance(records, list):
        raise RuntimeError(f"Evidence JSON must be a list of records: {path}")

    return records

def validate_minimum_record_keys(records, dataset_name):
    # Only these keys are needed for no-context LLM answering.
    required_keys = ["type", "question", "answer"]

    for i, rec in enumerate(records[:10]):
        if not isinstance(rec, dict):
            raise RuntimeError(f"{dataset_name} record {i} is not a dictionary.")

        missing = [k for k in required_keys if k not in rec]
        if missing:
            raise RuntimeError(f"{dataset_name} record {i} missing keys: {missing}")

for ds in DATASETS:
    records = load_evidence_records(ds["local_evidence_path"])
    validate_minimum_record_keys(records, ds["name"])
    ds["records"] = records

    print("=" * 100)
    print("Dataset:", ds["name"])
    print("Number of records:", len(records))
    print("First type:", records[0].get("type"))
    print("First question:", records[0].get("question"))
    print("First GT answer:", records[0].get("answer"))

Dataset: 2wikimultihopqa
Number of records: 1000
First type: bridge_comparison
First question: Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?
First GT answer: Kamakalawa
Dataset: hotpotqa
Number of records: 1000
First type: comparison
First question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
First GT answer: Bedknobs and Broomsticks


In [5]:
#cell5
# Start GPT-OSS-120B vLLM server.

def kill_process_tree(pid):
    # Kill a process and all children.
    try:
        parent = psutil.Process(int(pid))
        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass
        parent.kill()
        parent.wait(timeout=10)
        print("Killed process tree:", pid)
    except Exception:
        pass

def find_nvidia_library_dirs():
    # Find CUDA shared library directories installed by pip packages.
    dirs = []

    for base in site.getsitepackages():
        pattern = os.path.join(base, "nvidia", "*", "lib")
        for d in glob.glob(pattern):
            if os.path.isdir(d):
                dirs.append(d)

    unique_dirs = []
    seen = set()
    for d in dirs:
        if d not in seen:
            unique_dirs.append(d)
            seen.add(d)

    return unique_dirs

# Stop old PID from this notebook.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(old_pid)

# Stop leftover vLLM serve processes.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(3)

cmd = [
    "vllm", "serve", LLM_MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    "--generation-config", "vllm",
    "--trust-remote-code",

    # Let GPT-OSS / MXFP4 use the correct automatic dtype.
    "--dtype", "auto",
]

if KV_CACHE_DTYPE:
    cmd.extend(["--kv-cache-dtype", KV_CACHE_DTYPE])

if MAX_CUDAGRAPH_CAPTURE_SIZE:
    cmd.extend(["--max-cudagraph-capture-size", str(MAX_CUDAGRAPH_CAPTURE_SIZE)])

if not ENABLE_PREFIX_CACHING:
    cmd.append("--no-enable-prefix-caching")
else:
    cmd.append("--enable-prefix-caching")

server_env = os.environ.copy()

# Blackwell / CUDA 13 environment consistency.
server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"
server_env["TORCH_CUDA_ARCH_LIST"] = "12.0"

# Disable only the FlashInfer top-k/top-p sampler.
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "1" if USE_FLASHINFER_SAMPLER else "0"

# Optional fallback.
if DISABLE_FLASHINFER_COMPLETELY:
    server_env["VLLM_DISABLE_FLASHINFER"] = "1"

if ATTENTION_BACKEND:
    server_env["VLLM_ATTENTION_BACKEND"] = ATTENTION_BACKEND

# Add pip-installed NVIDIA library paths to help optional CUDA libraries resolve.
if ADD_NVIDIA_PIP_LIBS_TO_LD_LIBRARY_PATH:
    nvidia_lib_dirs = find_nvidia_library_dirs()
    old_ld_path = server_env.get("LD_LIBRARY_PATH", "")
    merged_ld_path = ":".join(nvidia_lib_dirs + ([old_ld_path] if old_ld_path else []))
    server_env["LD_LIBRARY_PATH"] = merged_ld_path

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in [
    "VLLM_MAIN_CUDA_VERSION",
    "TORCH_CUDA_ARCH_LIST",
    "VLLM_USE_FLASHINFER_SAMPLER",
    "VLLM_DISABLE_FLASHINFER",
    "VLLM_ATTENTION_BACKEND",
    "LD_LIBRARY_PATH",
]:
    value = server_env.get(k)
    if k == "LD_LIBRARY_PATH" and value:
        print(f"{k}={value[:500]}{'...' if len(value) > 500 else ''}")
    else:
        print(f"{k}={value}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Command:
vllm serve openai/gpt-oss-120b --host 0.0.0.0 --port 8000 --max-model-len 8192 --gpu-memory-utilization 0.95 --max-num-seqs 1 --max-num-batched-tokens 16384 --generation-config vllm --trust-remote-code --dtype auto --kv-cache-dtype fp8 --max-cudagraph-capture-size 2048 --no-enable-prefix-caching

Important environment variables:
VLLM_MAIN_CUDA_VERSION=13.0
TORCH_CUDA_ARCH_LIST=12.0
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_DISABLE_FLASHINFER=None
VLLM_ATTENTION_BACKEND=None
LD_LIBRARY_PATH=/usr/local/lib/python3.12/dist-packages/nvidia/nvshmem/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib:/usr/local/lib/python3.12/dist-packages/nvidia/nccl/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cusparse/lib:/usr/local/lib/python3.12/dist-packages/nvidia/curand/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cusolver/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvrtc/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvcc/lib:/usr/local/lib/python3.12/d...

In [6]:
#cell6
# Wait for vLLM server and create OpenAI-compatible client.

import requests

def tail_log(path, n=80):
    # Read last log lines.
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()
    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=5)
        if h.status_code == 200:
            m = requests.get(f"{BASE_URL}/models", timeout=10)
            if m.status_code == 200:
                ready = True
                model_info = m.json()["data"][0]
                print("vLLM server is ready.")
                print("Model:", model_info["id"])
                print("Max model len:", model_info.get("max_model_len"))
                break
    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")
        recent = tail_log(SERVER_LOG_PATH, n=12)
        if recent.strip():
            print(recent)
        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

client = OpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    timeout=3600,
)

# GPT-OSS is commonly evaluated with temperature=1.0 and top_p=1.0.
LLM_SAMPLING_KWARGS = {
    "temperature": 1.0,
    "top_p": 1.0,
    "presence_penalty": 0.0,
}

# Hide reasoning from the returned assistant content.
LLM_EXTRA_BODY = {
    "chat_template_kwargs": {
        "reasoning_effort": REASONING_EFFORT,
    },
    "include_reasoning": False,
}

print("OpenAI-compatible client is ready.")
print("Reasoning effort:", REASONING_EFFORT)
print("Sampling kwargs:", LLM_SAMPLING_KWARGS)
print("Extra body:", LLM_EXTRA_BODY)

Waiting... 0s
(APIServer pid=2889) INFO 06-14 09:31:21 [api_utils.py:339] 
(APIServer pid=2889) INFO 06-14 09:31:21 [api_utils.py:339]        █     █     █▄   ▄█
(APIServer pid=2889) INFO 06-14 09:31:21 [api_utils.py:339]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.22.1rc1.dev511+gc621af169
(APIServer pid=2889) INFO 06-14 09:31:21 [api_utils.py:339]   █▄█▀ █     █     █     █  model   openai/gpt-oss-120b
(APIServer pid=2889) INFO 06-14 09:31:21 [api_utils.py:339]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=2889) INFO 06-14 09:31:21 [api_utils.py:339] 
(APIServer pid=2889) INFO 06-14 09:31:21 [api_utils.py:273] non-default args: {'model_tag': 'openai/gpt-oss-120b', 'host': '0.0.0.0', 'model': 'openai/gpt-oss-120b', 'trust_remote_code': True, 'max_model_len': 8192, 'generation_config': 'vllm', 'gpu_memory_utilization': 0.95, 'kv_cache_dtype': 'fp8', 'enable_prefix_caching': False, 'max_num_batched_tokens': 16384, 'max_num_seqs': 1, 'max_cudagraph_capture_size': 2048}
(APIServer pid=2889) Warning

In [7]:
#cell7
# No-context answer prompt for GPT-OSS-120B.

ANSWER_SYSTEM_PROMPT = f"""
You answer questions directly and concisely in English.

Rules:
- Use only your own model knowledge.
- No context, evidence, passages, supports, or documents will be provided.
- If you do not know the answer, respond exactly: {UNKNOWN_ANSWER}
- Provide only the final answer.
- Do not include explanations.
- Do not include citations.
- Do not include markdown.
- Keep the answer under 6 words.
""".strip()

def clean_text_for_prompt(text):
    # Preserve text content, only normalize Python None.
    if text is None:
        return ""
    return str(text)

def build_answer_prompt(record):
    # Build a no-context prompt using only the question.
    question = clean_text_for_prompt(record.get("question", ""))

    prompt = f"""Given the following question, create a final answer in English to the question.

QUESTION: {question}

ANSWER: [Please provide only the answer and keep the answer less than 6 words. If you do not know the answer, write exactly: {UNKNOWN_ANSWER}.]
""".strip()

    return prompt

# Preview one prompt from each dataset.
for ds in DATASETS:
    print("=" * 100)
    print("Dataset:", ds["name"])
    preview_prompt = build_answer_prompt(ds["records"][0])
    print(preview_prompt)
    print("Prompt preview length in characters:", len(preview_prompt))

Dataset: 2wikimultihopqa
Given the following question, create a final answer in English to the question.

QUESTION: Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?

ANSWER: [Please provide only the answer and keep the answer less than 6 words. If you do not know the answer, write exactly: I don't know.]
Prompt preview length in characters: 312
Dataset: hotpotqa
Given the following question, create a final answer in English to the question.

QUESTION: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?

ANSWER: [Please provide only the answer and keep the answer less than 6 words. If you do not know the answer, write exactly: I don't know.]
Prompt preview length in characters: 324


In [8]:
#cell8
# Answer cleanup and GPT-OSS call helpers.

def strip_code_fence(text):
    # Remove markdown code fences.
    text = (text or "").strip()
    text = re.sub(r"^```(?:json|text)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

def strip_think_blocks(text):
    # Remove raw thinking blocks if returned.
    return re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL).strip()

def normalize_unknown_answer(text):
    # Normalize common unknown-answer variants.
    normalized = re.sub(r"\s+", " ", (text or "").strip().lower())

    unknown_variants = {
        "",
        "unknown",
        "not known",
        "i don't know",
        "i do not know",
        "i dont know",
        "don't know",
        "do not know",
        "cannot determine",
        "can't determine",
        "not sure",
        "information not available",
        "not available",
        "insufficient information",
        "not enough information",
    }

    if normalized in unknown_variants:
        return UNKNOWN_ANSWER

    return text

def remove_answer_prefixes(text):
    # Remove accidental answer prefixes.
    text = re.sub(r"^\s*answer\s*:\s*", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"^\s*final answer\s*:\s*", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"^\s*the answer is\s+", "", text, flags=re.IGNORECASE).strip()
    return text

def count_words(text):
    # Count whitespace-separated words.
    return len(re.findall(r"\S+", text or ""))

def truncate_to_max_words(text, max_words=MAX_ANSWER_WORDS):
    # Enforce the requested answer length as a last-resort cleanup.
    words = re.findall(r"\S+", text or "")
    if len(words) <= max_words:
        return text.strip()
    return " ".join(words[:max_words]).strip()

def get_message_content(msg):
    # GPT-OSS/vLLM may expose content in different shapes.
    content = getattr(msg, "content", None) or ""

    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, dict):
                parts.append(part.get("text") or part.get("content") or "")
            else:
                parts.append(str(part))
        content = "".join(parts)

    return content

def clean_model_answer(text, force_truncate=False):
    # Clean a plain-text model answer.
    answer = strip_think_blocks(text)
    answer = strip_code_fence(answer)
    answer = answer.strip()

    # Keep only the first non-empty line if the model returns multiple lines.
    lines = [line.strip() for line in answer.splitlines() if line.strip()]
    answer = lines[0] if lines else ""

    answer = remove_answer_prefixes(answer)

    # Remove accidental surrounding quotes.
    if len(answer) >= 2 and answer[0] == answer[-1] and answer[0] in ['"', "'"]:
        answer = answer[1:-1].strip()

    answer = re.sub(r"\s+", " ", answer).strip()
    answer = normalize_unknown_answer(answer)

    if not answer:
        answer = UNKNOWN_ANSWER

    if force_truncate and answer != UNKNOWN_ANSWER:
        answer = truncate_to_max_words(answer, MAX_ANSWER_WORDS)

    return answer

def usage_to_dict(usage):
    # Convert OpenAI-compatible usage object to dictionary.
    if usage is None:
        return None

    try:
        return usage.model_dump()
    except Exception:
        try:
            return dict(usage)
        except Exception:
            return None

def get_completion_tokens_from_usage(usage):
    # Read completion token count if vLLM/OpenAI-compatible usage provides it.
    usage_dict = usage_to_dict(usage)
    if not usage_dict:
        return None
    return usage_dict.get("completion_tokens")

def call_llm_answer(prompt, max_retries=2):
    # Call GPT-OSS-120B and return one short plain-text answer.
    last_content = None
    last_error = None

    base_messages = [
        {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    for attempt in range(max_retries + 1):
        if attempt == 0:
            messages = base_messages
        else:
            messages = [
                {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": (
                        prompt
                        + "\n\nYour previous answer was not valid for the requested format. "
                        + f"Return only one answer under 6 words. If unknown, return exactly: {UNKNOWN_ANSWER}\n"
                        + f"Previous answer: {last_content}"
                    ),
                },
            ]

        try:
            response = client.chat.completions.create(
                model=LLM_MODEL_NAME,
                messages=messages,
                max_tokens=ANSWER_MAX_TOKENS if attempt == 0 else ANSWER_RETRY_MAX_TOKENS,
                **LLM_SAMPLING_KWARGS,
                extra_body=LLM_EXTRA_BODY,
            )

            msg = response.choices[0].message
            content = get_message_content(msg)
            last_content = content

            answer = clean_model_answer(content, force_truncate=False)
            completion_tokens = get_completion_tokens_from_usage(response.usage)
            usage_dict = usage_to_dict(response.usage)

            # Retry if the model returned a verbose answer.
            if answer != UNKNOWN_ANSWER and count_words(answer) > MAX_ANSWER_WORDS:
                last_error = f"Answer has too many words: {count_words(answer)}"
                if attempt < max_retries:
                    continue

                answer = clean_model_answer(answer, force_truncate=True)

            return {
                "response": answer,
                "raw_content": content,
                "usage": usage_dict,
                "completion_tokens": completion_tokens,
                "attempt": attempt,
                "error": last_error,
            }

        except Exception as e:
            last_error = repr(e)
            if attempt >= max_retries:
                break
            continue

    return {
        "response": UNKNOWN_ANSWER,
        "raw_content": last_content,
        "usage": None,
        "completion_tokens": None,
        "attempt": "fallback",
        "error": last_error,
    }

In [9]:
#cell9
# Output I/O and resume helpers.

def atomic_write_json(path, data):
    # Atomic JSON write.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    os.replace(tmp, path)

def load_existing_outputs(path):
    # Load existing output list for resume.
    path = Path(path)

    if not path.exists():
        return []

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return []

    if not isinstance(data, list):
        return []

    return data

def save_outputs(path, outputs):
    # Save output records.
    atomic_write_json(path, outputs)

def make_output_record(record, llm_result):
    # Build the requested final output schema only.
    return {
        "type": record.get("type"),
        "question": record.get("question"),
        "gt": record.get("answer"),
        "response": llm_result.get("response", UNKNOWN_ANSWER),
    }

def make_error_output_record(record, error):
    # Build an error fallback record while preserving the requested schema.
    print("Error while processing record:")
    print(error)
    print(traceback.format_exc())

    return {
        "type": record.get("type"),
        "question": record.get("question"),
        "gt": record.get("answer"),
        "response": UNKNOWN_ANSWER,
    }

def output_record_is_complete(output_record, source_record):
    # Check whether an existing output record can be skipped.
    if not isinstance(output_record, dict):
        return False

    required_keys = ["type", "question", "gt", "response"]
    if any(k not in output_record for k in required_keys):
        return False

    if output_record.get("question") != source_record.get("question"):
        return False

    if not output_record.get("response"):
        return False

    return True

In [10]:
#cell10
# Run no-context answer generation for one dataset.

def run_dataset(ds):
    # Process one dataset independently and save one separate JSON file.
    dataset_name = ds["name"]
    records = ds["records"]
    output_path = ds["drive_answer_path"]

    start_idx = int(ANSWER_START_INDEX)
    end_idx = len(records) if ANSWER_END_INDEX is None else int(ANSWER_END_INDEX)
    run_records = records[start_idx:end_idx]

    existing_outputs = load_existing_outputs(output_path)

    print("=" * 100)
    print("Dataset:", dataset_name)
    print("Total records:", len(records))
    print("Run range:", start_idx, "to", end_idx)
    print("Existing output records:", len(existing_outputs))
    print("Output path:", output_path)

    progress = tqdm(
        enumerate(run_records, start=start_idx),
        total=len(run_records),
        desc=f"Generating answers: {dataset_name}",
        dynamic_ncols=True,
    )

    outputs = existing_outputs

    for source_index, record in progress:
        # Skip completed records by list position.
        if source_index < len(outputs):
            old_output = outputs[source_index]
            if output_record_is_complete(old_output, record):
                progress.set_postfix({
                    "dataset": dataset_name,
                    "source_index": source_index,
                    "status": "skipped",
                    "saved": len(outputs),
                })
                continue

        try:
            prompt = build_answer_prompt(record)
            llm_result = call_llm_answer(prompt)
            output_record = make_output_record(record, llm_result)
            status = "ok"

        except Exception as e:
            output_record = make_error_output_record(record, e)
            status = "error"

        # Keep output order identical to input order.
        if source_index < len(outputs):
            outputs[source_index] = output_record
        elif source_index == len(outputs):
            outputs.append(output_record)
        else:
            # This should not happen when ANSWER_START_INDEX is 0.
            # Fill missing positions with fallback records to preserve list alignment.
            while len(outputs) < source_index:
                missing_record = records[len(outputs)]
                outputs.append({
                    "type": missing_record.get("type"),
                    "question": missing_record.get("question"),
                    "gt": missing_record.get("answer"),
                    "response": UNKNOWN_ANSWER,
                })
            outputs.append(output_record)

        if (source_index + 1) % SAVE_EVERY_N == 0:
            save_outputs(output_path, outputs)

        if (source_index + 1) % CLEAR_CACHE_EVERY_N == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        progress.set_postfix({
            "dataset": dataset_name,
            "source_index": source_index,
            "status": status,
            "saved": len(outputs),
        })

    save_outputs(output_path, outputs)

    print("Finished dataset:", dataset_name)
    print("Saved records:", len(outputs))
    print("Output file:", output_path)

    return outputs

In [11]:
#cell11
# Run both datasets separately.

all_outputs = {}

for ds in DATASETS:
    # Each dataset is processed independently.
    # The output of one dataset is never mixed with the other dataset.
    outputs = run_dataset(ds)
    all_outputs[ds["name"]] = outputs

print("=" * 100)
print("All datasets finished.")

for ds in DATASETS:
    output_path = ds["drive_answer_path"]
    print(ds["name"], "->", output_path)

Dataset: 2wikimultihopqa
Total records: 1000
Run range: 0 to 1000
Existing output records: 0
Output path: /content/drive/MyDrive/final_project/LLM/2wikimultihopqa_gpt_oss_120b_answers.json


Generating answers: 2wikimultihopqa:   0%|          | 0/1000 [00:00<?, ?it/s]

Finished dataset: 2wikimultihopqa
Saved records: 1000
Output file: /content/drive/MyDrive/final_project/LLM/2wikimultihopqa_gpt_oss_120b_answers.json
Dataset: hotpotqa
Total records: 1000
Run range: 0 to 1000
Existing output records: 0
Output path: /content/drive/MyDrive/final_project/LLM/hotpotqa_gpt_oss_120b_answers.json


Generating answers: hotpotqa:   0%|          | 0/1000 [00:00<?, ?it/s]

Finished dataset: hotpotqa
Saved records: 1000
Output file: /content/drive/MyDrive/final_project/LLM/hotpotqa_gpt_oss_120b_answers.json
All datasets finished.
2wikimultihopqa -> /content/drive/MyDrive/final_project/LLM/2wikimultihopqa_gpt_oss_120b_answers.json
hotpotqa -> /content/drive/MyDrive/final_project/LLM/hotpotqa_gpt_oss_120b_answers.json


In [12]:
#cell12
# Inspect saved answers from both output files.

for ds in DATASETS:
    output_path = ds["drive_answer_path"]

    with open(output_path, "r", encoding="utf-8") as f:
        saved_answers = json.load(f)

    print("=" * 100)
    print("Dataset:", ds["name"])
    print("Saved answers:", len(saved_answers))
    print("Output path:", output_path)

    for rec in saved_answers[:5]:
        print("-" * 100)
        print("type:", rec.get("type"))
        print("question:", rec.get("question"))
        print("gt:", rec.get("gt"))
        print("response:", rec.get("response"))

Dataset: 2wikimultihopqa
Saved answers: 1000
Output path: /content/drive/MyDrive/final_project/LLM/2wikimultihopqa_gpt_oss_120b_answers.json
----------------------------------------------------------------------------------------------------
type: bridge_comparison
question: Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?
gt: Kamakalawa
response: I don't know
----------------------------------------------------------------------------------------------------
type: compositional
question: Where did Prince Gustav Of Thurn And Taxis (1848–1914)'s mother die?
gt: Meran
response: I don't know
----------------------------------------------------------------------------------------------------
type: bridge_comparison
question: Which film has the director died later, A Light Woman or Our Mother'S House?
gt: Our Mother'S House
response: I don't know
----------------------------------------------------------------------------------------------------
type: bridge_